In [ ]:
import numpy as np
from scipy.stats import beta, wasserstein_distance

# -------------------------------
# Setup
# -------------------------------

# Bin boundaries
bins = np.linspace(0, 1, 11)  # 5 bins: [0,0.2], [0.2,0.4], ...
question_conf_dists = [
    (6, 4), (3, 2), (8, 2), (5, 5), (2, 8),
    (7, 3), (4, 6), (9, 1), (1, 9), (6, 6),
    (5, 3), (3, 5), (7, 2), (2, 7), (8, 4)
]

# Generate correctness labels randomly (0 or 1)
question_correctness = np.random.randint(0, 2, size=len(question_conf_dists)).tolist()
num_samples = 100000

# Sample question-level confidence distributions
samples_list = [beta.rvs(a, b, size=num_samples) for a, b in question_conf_dists]

# Compute **mean confidence** per question for traditional ECE
mean_confidences = [np.mean(samples) for samples in samples_list]

# -------------------------------
# Soft-bin Wasserstein dECE
# -------------------------------

dECE_bins = []
bin_total_weights = []

for m in range(len(bins)-1):
    s_min, s_max = bins[m], bins[m+1]

    weighted_W = []
    total_weight = 0.0
    weighted_correct_sum = 0.0  # soft-bin accuracy numerator

    # Step 1: compute soft membership and accumulate
    for samples, y in zip(samples_list, question_correctness):
        w = np.mean((samples >= s_min) & (samples < s_max))
        if w == 0:
            continue
        weighted_correct_sum += w * y
        total_weight += w

    # Step 2: soft-bin weighted accuracy
    if total_weight > 0:
        bin_acc = weighted_correct_sum / total_weight
    else:
        bin_acc = 0.0

    # Step 3: per-question Wasserstein distance
    for samples in samples_list:
        w = np.mean((samples >= s_min) & (samples < s_max))
        if w == 0:
            continue
        samples_bin = samples[(samples >= s_min) & (samples < s_max)]
        acc_samples = np.full(len(samples_bin), bin_acc)
        W = wasserstein_distance(samples_bin, acc_samples)
        weighted_W.append(w * W)

    # Step 4: weighted-average dECE for bin
    if total_weight > 0:
        dECE_bin = np.sum(weighted_W) / total_weight
    else:
        dECE_bin = 0.0

    dECE_bins.append(dECE_bin)
    bin_total_weights.append(total_weight)

# -------------------------------
# Dataset-level soft-bin dECE
# -------------------------------
bin_total_weights = np.array(bin_total_weights)
dECE_bins = np.array(dECE_bins)
dataset_dECE = np.sum(dECE_bins * bin_total_weights) / bin_total_weights.sum()

print("Soft-bin Wasserstein dECE per bin:", dECE_bins)
print("Total soft membership per bin:", bin_total_weights)
print("Dataset-level dECE (weighted by bin mass):", dataset_dECE)

# -------------------------------
# Traditional ECE (hard binning)
# -------------------------------
hard_bin_indices = np.digitize(mean_confidences, bins) - 1  # map to bins 0..M-1
M = len(bins) - 1
ece_bins = []
ece_bin_counts = []

for m in range(M):
    # questions in this bin
    in_bin = [i for i, b in enumerate(hard_bin_indices) if b == m]
    N_m = len(in_bin)
    ece_bin_counts.append(N_m)
    if N_m == 0:
        ece_bins.append(0.0)
        continue
    # mean confidence and mean accuracy
    c_bar = np.mean([mean_confidences[i] for i in in_bin])
    a_bar = np.mean([question_correctness[i] for i in in_bin])
    ece_bins.append(abs(c_bar - a_bar))

# weighted average by number of questions per bin
N_total = len(question_conf_dists)
traditional_ECE = np.sum([N_m / N_total * e for N_m, e in zip(ece_bin_counts, ece_bins)])

print("Traditional ECE per bin:", ece_bins)
print("Traditional ECE:", traditional_ECE)
